05_latent_gig_eligible

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import brier_score_loss

SEED = 42
OUT_DIR = Path("outputs")

stage2 = pd.read_pickle(OUT_DIR / "stage2_working_women.pkl")
outside = pd.read_pickle(OUT_DIR / "outside_lf_women.pkl").reset_index(drop=True)
seeking = pd.read_pickle(OUT_DIR / "seeking_women.pkl")
specs = joblib.load(OUT_DIR / "stage2_model_specs.joblib")
FEATURES, CAT, NUM = specs["features"], specs["cat"], specs["num"]

VL = pd.read_pickle(OUT_DIR / "value_labels.pkl")
SRC = {"sector": "b1q3_perv1", "social_group": "b3q4_hhv1", "religion": "b3q3_hhv1",
       "state": "state_perv1", "marital": "b4q7_perv1"}
def lab(series, var):
    """Turn codes into SPSS value labels. '07' -> '7' and '7.0' -> '7', so state codes 01-09 match their labels."""
    m = {}
    for k, v in VL.get(SRC.get(var, var), {}).items():
        try:
            m[str(int(float(k)))] = v
        except Exception:
            m[str(k)] = v
    s = (series.astype(str)
               .str.replace(r"\.0$", "", regex=True)
               .str.replace(r"^0+(?=\d)", "", regex=True))
    return s.map(m).fillna(s)

def pop_weight(df):
    """PLFS weight: multiplier/100 if NSS == NSC, else multiplier/200. VERIFY in the PLFS manual."""
    need = ["NSS_perv1", "NSC_perv1", "mult_perv1"]
    if not set(need) <= set(df.columns):
        print("Weight columns missing: using weight = 1 (sample counts only)")
        return pd.Series(1.0, index=df.index)
    nss, nsc, mult = [pd.to_numeric(df[c], errors="coerce") for c in need]
    return pd.Series(np.where(nss == nsc, mult / 100, mult / 200), index=df.index).fillna(0)

outside["w"] = pop_weight(outside)
print("Outside LF women:", len(outside), "| weighted total: %.1f million" % (outside["w"].sum() / 1e6))
print("Stage 2 specs saved for:", [k for k in specs if k in ("main", "other")],
      "| main model:", specs["main"]["name"])

def make_pipe(model):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), CAT),
        ("num", StandardScaler(), NUM)])
    return Pipeline([("pre", pre), ("model", model)])

In [ ]:
W_SCALE = 0.25   # four pooled quarters. CONFIRM with the check below; if unsure set to None and report counts and shares only
outside["w"] = pop_weight(outside) * (W_SCALE if W_SCALE else 1.0)
print("Weighted women outside the labour force: %.1f million (scale used: %s)" % (outside["w"].sum() / 1e6, W_SCALE))

In [ ]:
import pyreadstat
DATA_PATH = Path("data") / "PERV MER COPY - Copy.sav"
wdf, _ = pyreadstat.read_sav(DATA_PATH, usecols=["NSS_perv1", "NSC_perv1", "mult_perv1"])
w_all = pop_weight(wdf)
print("All persons, raw weights:    %.0f million" % (w_all.sum() / 1e6))
print("All persons, divided by 4:   %.0f million  (should be close to India's population, roughly 1,400 million)" % (w_all.sum() / 4 / 1e6))

In [ ]:
TARGET_PRECISION = 0.25    # decided in advance: minimum precision among working women

def latent_analysis(key, target):
    d = stage2[stage2[target].notna()].reset_index(drop=True)
    X, y, g = d[FEATURES], d[target].astype(int), d["HHID"]
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

    # 1. Out-of-fold scores for working women (each from a model that never saw her household)
    oof = cross_val_predict(clone(specs[key]["pipe"]), X, y, groups=g, cv=cv, method="predict_proba")[:, 1]

    # 2. Calibrate raw scores into probabilities (isotonic, fitted on working women)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(oof, y)
    cal_oof = iso.predict(oof)

    # 3. Final model on all working women, then score women outside the labour force
    final = clone(specs[key]["pipe"]).fit(X, y)
    raw_out = final.predict_proba(outside[FEATURES])[:, 1]
    cal_out = iso.predict(raw_out)

    # 4. High-resemblance cutoffs, defined on working women BEFORE looking at non-participants
    cut_decile = np.quantile(oof, 0.90)
    prec_dec = y[oof >= cut_decile].mean()
    cut_prec = None
    for q in np.arange(0.50, 0.995, 0.005):
        c = np.quantile(oof, q)
        if y[oof >= c].mean() >= TARGET_PRECISION and (oof >= c).sum() >= 200:
            cut_prec = c; break          # lowest cutoff that reaches the target precision
    prec_at_target = y[oof >= cut_prec].mean() if cut_prec is not None else np.nan
    recall_at_target = y[oof >= cut_prec].sum() / y.sum() if cut_prec is not None else np.nan

    return dict(key=key, target=target, n_work=len(d), prevalence=float(y.mean()),
                oof=oof, cal_oof=cal_oof, y=y, raw_out=raw_out, cal_out=cal_out,
                cut_decile=cut_decile, prec_decile=float(prec_dec),
                cut_prec=cut_prec, prec_at_target=float(prec_at_target), recall_at_target=float(recall_at_target),
                brier_raw=brier_score_loss(y, oof), brier_cal=brier_score_loss(y, cal_oof))

In [ ]:
main = latent_analysis("main", "y_gig_vs_trad")

print(f"Working women used: {main['n_work']:,} | gig-proxy share {main['prevalence']:.3f}")
print(f"Brier score (lower is better): raw {main['brier_raw']:.3f} -> calibrated {main['brier_cal']:.3f}")

cal_table = (pd.DataFrame({"raw": main["oof"], "calibrated": main["cal_oof"], "actual": main["y"]})
             .assign(decile=lambda t: pd.qcut(t["raw"], 10, labels=False, duplicates="drop") + 1)
             .groupby("decile")[["raw", "calibrated", "actual"]].mean().round(3))
print("\nCalibration by score decile among working women (calibrated should track actual):")
display(cal_table)

print(f"\nCutoff A (top decile of working women): raw score >= {main['cut_decile']:.3f}; "
      f"precision among working women {main['prec_decile']:.3f}")
if main["cut_prec"] is not None:
    print(f"Cutoff B (precision target {TARGET_PRECISION:.0%}): raw score >= {main['cut_prec']:.3f}; "
          f"precision {main['prec_at_target']:.3f}, recall {main['recall_at_target']:.3f}")
else:
    print("Cutoff B: no cutoff reaches the target precision; report cutoff A only.")

In [ ]:
# Domain model: how much does each non-participant look like a working woman?
work_X = stage2[FEATURES]
dom_X = pd.concat([work_X, outside[FEATURES]], ignore_index=True)
dom_y = np.r_[np.ones(len(work_X)), np.zeros(len(outside))]
dom = make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced")).fit(dom_X, dom_y)
p_work_w = dom.predict_proba(work_X)[:, 1]
p_work_o = dom.predict_proba(outside[FEATURES])[:, 1]
support_cut = np.quantile(p_work_w, 0.01)          # 1st percentile among working women
outside["on_support"] = p_work_o >= support_cut
print("Non-participants within the range of working women (on support): %.1f%%" % (100 * outside["on_support"].mean()))

def add_scores(res, suffix=""):
    outside[f"raw{suffix}"] = res["raw_out"]
    outside[f"cal{suffix}"] = res["cal_out"]
    outside[f"flag_decile{suffix}"] = res["raw_out"] >= res["cut_decile"]
    outside[f"flag_prec{suffix}"] = (res["raw_out"] >= res["cut_prec"]) if res["cut_prec"] is not None else False
add_scores(main)
outside["cal"], outside["flag_decile"], outside["flag_prec"] = outside["cal"], outside["flag_decile"], outside["flag_prec"]

def headline(df, flag):
    f = df[flag]
    return {"n_outside": len(df), "n_flagged": int(f.sum()), "share_flagged": f.mean(),
            "weighted_millions_flagged": (df["w"] * f).sum() / 1e6,
            "weighted_share_flagged": (df["w"] * f).sum() / df["w"].sum(),
            "expected_gig_like_n (sum of calibrated p)": df["cal"].sum(),
            "expected_gig_like_weighted_millions": (df["w"] * df["cal"]).sum() / 1e6,
            "mean_calibrated_p": df["cal"].mean()}

rows = {}
for label, df in [("All outside LF", outside), ("On support only", outside[outside["on_support"]])]:
    rows[f"{label} | top-decile cutoff"] = headline(df, "flag_decile")
    rows[f"{label} | precision-target cutoff"] = headline(df, "flag_prec")
headline_tbl = pd.DataFrame(rows).T
display(headline_tbl.round(3)); headline_tbl.round(4).to_csv(OUT_DIR / "latent_headline.csv")

# Women seeking work are already in the labour force; shown separately for reference
seek_cal = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)  # placeholder to avoid refit confusion
print("\nSeeking-work women (not part of the latent estimate):", len(seeking))

In [ ]:
def breakdown(df, col, flag):
    k = lab(df[col], col).values
    t = (df.assign(_k=k, _f=df[flag].astype(int), _wf=df["w"] * df[flag].astype(int), _wp=df["w"] * df["cal"])
           .groupby("_k").agg(n_outside=("HHID", "size"), n_flagged=("_f", "sum"),
                              expected_n=("cal", "sum"), mean_cal_p=("cal", "mean"),
                              w_outside=("w", "sum"), w_flagged=("_wf", "sum"), w_expected=("_wp", "sum")))
    t["share_flagged"] = t["n_flagged"] / t["n_outside"]
    t["w_share_flagged"] = t["w_flagged"] / t["w_outside"]
    t[["w_outside", "w_flagged", "w_expected"]] = t[["w_outside", "w_flagged", "w_expected"]] / 1e3   # thousands
    return t.round(3)

FLAG = "flag_decile"      # change to "flag_prec" to use the precision-target cutoff
for col in ["sector", "social_group"]:
    print(f"\n--- by {col} (weighted columns in thousands of women)")
    display(breakdown(outside, col, FLAG))

st = breakdown(outside, "state", FLAG).sort_values("mean_cal_p", ascending=False)
print("\n--- by state, ranked by mean calibrated probability")
display(st)
st.to_csv(OUT_DIR / "latent_by_state.csv")

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].hist(main["cal_oof"], bins=40, alpha=0.6, density=True, label="Working women (out-of-fold)", color="#1F3A5F")
ax[0].hist(outside["cal"], bins=40, alpha=0.6, density=True, label="Outside labour force", color="#B5483A")
ax[0].set_xlabel("Calibrated probability of resembling gig-proxy work"); ax[0].legend(); ax[0].set_title("Score distributions")
st["mean_cal_p"].sort_values().plot.barh(ax=ax[1], color="#2E5A88"); ax[1].set_title("Mean calibrated probability by state")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_latent.png", dpi=150); plt.show()

In [ ]:
hi = outside[outside["flag_decile"]]; lo = outside[~outside["flag_decile"]]
prof = pd.DataFrame({
    "high_resemblance": [hi[c].mean() for c in NUM],
    "other_outside_LF": [lo[c].mean() for c in NUM],
    "working_gig_proxy": [stage2.loc[stage2["y_gig_vs_trad"] == 1, c].mean() for c in NUM],
}, index=NUM).round(2)
prof.loc["rural_share"] = [(hi["sector"] == "1").mean(), (lo["sector"] == "1").mean(),
                           (stage2.loc[stage2["y_gig_vs_trad"] == 1, "sector"] == "1").mean()]
display(prof)
for col in ["edu_group", "marital", "social_group"]:
    k = lab(hi[col], col) if col in SRC else hi[col]
    print(f"\nHigh-resemblance women by {col} (share):")
    display(k.value_counts(normalize=True).round(3))
prof.to_csv(OUT_DIR / "latent_profile.csv")

In [ ]:
alt = latent_analysis("other", "y_gig_vs_other")
add_scores(alt, "_alt")
rob = {}
for name, res, flag, cal in [("MAIN (gig vs traditional)", main, "flag_decile", "cal"),
                             ("ROBUSTNESS (gig vs all others)", alt, "flag_decile_alt", "cal_alt")]:
    f = outside[flag]
    rob[name] = {"top-decile cutoff precision (working women)": res["prec_decile"],
                 "n_flagged": int(f.sum()), "weighted_millions_flagged": (outside["w"] * f).sum() / 1e6,
                 "expected_gig_like_n": outside[cal].sum(),
                 "expected_gig_like_weighted_millions": (outside["w"] * outside[cal]).sum() / 1e6}
display(pd.DataFrame(rob).T.round(3))
pd.DataFrame(rob).T.round(4).to_csv(OUT_DIR / "latent_robustness.csv")

both = (outside["flag_decile"] & outside["flag_decile_alt"]).sum()
print(f"Flagged by both models: {both:,} ({both / max(outside['flag_decile'].sum(), 1):.1%} of the main flags)")
outside.drop(columns=["w"]).to_pickle(OUT_DIR / "outside_scored.pkl")

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score

def latent_variant(key, target, drop):
    """Stage 2 model and scoring of non-participants with some area variables removed."""
    cats = [c for c in CAT if c not in drop]
    feats = cats + NUM
    d_ = stage2[stage2[target].notna()].reset_index(drop=True)
    X, y_, g_ = d_[feats], d_[target].astype(int), d_["HHID"]
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), cats),
        ("num", StandardScaler(), NUM)])
    pipe = Pipeline([("pre", pre), ("model", clone(specs[key]["pipe"].named_steps["model"]))])
    cv_ = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_ = cross_val_predict(clone(pipe), X, y_, groups=g_, cv=cv_, method="predict_proba")[:, 1]
    iso_ = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(oof_, y_)
    final_ = clone(pipe).fit(X, y_)
    raw_out = final_.predict_proba(outside[feats])[:, 1]
    return {"y": y_, "oof": oof_, "raw_out": raw_out, "cal_out": iso_.predict(raw_out),
            "cut": np.quantile(oof_, 0.90)}

ns = latent_variant("main", "y_gig_vs_trad", ["state"])
na = latent_variant("main", "y_gig_vs_trad", ["state", "sector"])
for tag, r_ in [("nostate", ns), ("noarea", na)]:
    outside[f"cal_{tag}"] = r_["cal_out"]
    outside[f"flag_{tag}"] = r_["raw_out"] >= r_["cut"]

def sens_row(label, y_, oof_, cal_col, flag_col):
    return {"model": label,
            "ROC-AUC (working women, out-of-fold)": roc_auc_score(y_, oof_),
            "PR-AUC": average_precision_score(y_, oof_),
            "n flagged": int(outside[flag_col].sum()),
            "weighted flagged (millions)": (outside["w"] * outside[flag_col]).sum() / 1e6,
            "expected gig-like (n)": outside[cal_col].sum(),
            "expected gig-like (weighted millions)": (outside["w"] * outside[cal_col]).sum() / 1e6}

sens = pd.DataFrame([
    sens_row("A. With state (main)", main["y"], main["oof"], "cal", "flag_decile"),
    sens_row("B. No state", ns["y"], ns["oof"], "cal_nostate", "flag_nostate"),
    sens_row("C. No state, no sector", na["y"], na["oof"], "cal_noarea", "flag_noarea"),
]).set_index("model").round(3)
display(sens); sens.to_csv(OUT_DIR / "latent_sensitivity_no_state.csv")

def state_mean(col):
    return outside.assign(_s=lab(outside["state"], "state").values).groupby("_s")[col].mean()

s_main, s_ns = state_mean("cal"), state_mean("cal_nostate")
print("Rank correlation of state mean scores, with vs without state: %.3f" % spearmanr(s_main, s_ns.loc[s_main.index])[0])
cmp_state = pd.DataFrame({"with_state": s_main, "no_state": s_ns}).sort_values("with_state", ascending=False).round(3)
display(pd.concat([cmp_state.head(8), cmp_state.tail(8)]))
cmp_state.to_csv(OUT_DIR / "latent_state_with_vs_without_state.csv")

sec = lab(outside["sector"], "sector").values
display(pd.DataFrame({"with_state": outside.groupby(sec)["flag_decile"].mean(),
                      "no_state": outside.groupby(sec)["flag_nostate"].mean(),
                      "no_state_no_sector": outside.groupby(sec)["flag_noarea"].mean()}).round(3))
both = (outside["flag_decile"] & outside["flag_nostate"]).sum()
print(f"Flagged by both the main and the no-state model: {both:,} ({both / max(outside['flag_decile'].sum(), 1):.1%} of main flags)")
outside.drop(columns=["w"]).to_pickle(OUT_DIR / "outside_scored.pkl")